# Лабораторная работа №1. Описательная статистика и первые гипотезы

## Контекст задачи

Вы — аналитик данных в страховой компании **MedGuard**, которая продаёт полисы добровольного медицинского страхования (ДМС) в четырёх регионах США.

Продакт-менеджер (PM) готовит пересмотр тарифов и маркетинговой стратегии на следующий год. Перед тем как обсуждать что-либо с финансовым отделом, ему нужен **фактический портрет клиентской базы** и проверка нескольких утверждений, которые сейчас в компании принимаются «на веру».

Из внутренней переписки PM:

> «У нас есть выгрузка по 1 338 застрахованным за прошлый год. Мне нужно понять, кто наши клиенты, сколько мы на них реально тратим, и правда ли то, что говорят маркетинг и финансы. Ответы нужны с цифрами, а не с ощущениями. Каждый вывод — с обоснованием, почему ты в нём уверен».

**Ваша задача** — пройти по данным от описательной статистики до проверки гипотез и в конце написать короткий отчёт для PM.

---

## Датасет

Используем открытый датасет **Medical Cost Personal Dataset** (страховые выплаты по клиентам).

**Скачайте файл самостоятельно** по ссылке и положите его в одну папку с этим ноутбуком:

- https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv (правая кнопка → «Сохранить как…» → `insurance.csv`)
- Зеркало на Kaggle: https://www.kaggle.com/datasets/mirichoi0218/insurance

Описание колонок:

| Колонка | Описание |
|---|---|
| `age` | возраст застрахованного, лет |
| `sex` | пол: `male` / `female` |
| `bmi` | индекс массы тела (кг/м²) |
| `children` | количество детей, покрытых полисом |
| `smoker` | курит ли клиент: `yes` / `no` |
| `region` | регион проживания: `northeast`, `northwest`, `southeast`, `southwest` |
| `charges` | сумма страховых выплат за год, USD |

---

## Правила выполнения

1. Все расчёты — в коде (`numpy`, `pandas`, `scipy.stats`, `matplotlib`/`seaborn`), как в лекции.
2. После каждого задания — **вывод СВОИМИ словами** в markdown-ячейке: что посчитали, что это значит для бизнеса.
3. Уровень значимости во всех проверках гипотез: $\alpha = 0.05$.
4. Для проверки гипотез используем **z-критерий для среднего** и доверительные интервалы, как в лекции (выборка большая, $N > 30$, поэтому вместо $\sigma$ ГС берём $sd$ выборки).
5. Ячейки с `# ваш код` — заполняете вы. Ячейки с `**Вывод:**` — тоже вы.

In [ ]:
import math
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid")
np.random.seed(42)  # чтобы результаты со случайностью воспроизводились

df = pd.read_csv("insurance.csv")
print(df.shape)
df.head()

## Задание 1. Знакомство с данными и типы переменных

PM просит для начала «просто описать, что у нас за таблица». Но прежде чем считать хоть что-то, нужно понять, **с какими типами переменных** мы работаем — от этого зависит, какие меры и графики допустимы.

**1.1.** Выведите общую информацию о таблице: количество строк и столбцов, типы данных, наличие пропусков (`df.info()`, `df.isna().sum()`, `df.describe(include="all")`).

**1.2.** Заполните таблицу ниже: для каждой колонки определите тип переменной по классификации из лекции
(количественная непрерывная / количественная дискретная / номинативная / ранговая) и **обоснуйте в одном предложении**.

**1.3.** Ответьте: для какой из колонок будет **ошибкой** считать среднее значение, даже если pandas это позволит? Почему?

In [ ]:
# ваш код: общая информация о данных


**Вывод (заполните таблицу):**

| Колонка | Тип переменной | Обоснование |
|---|---|---|
| `age` | | |
| `sex` | | |
| `bmi` | | |
| `children` | | |
| `smoker` | | |
| `region` | | |
| `charges` | | |

**Ответ на 1.3:** 

## Задание 2. Генеральная совокупность и выборка

Финансовый отдел предлагает сэкономить: вместо обработки всей базы делать выводы по **выборке из 100 клиентов**. PM спрашивает, насколько это безопасно и как правильно отобрать этих 100 человек.

В рамках задания считаем, что наша таблица из 1 338 клиентов — это **генеральная совокупность (ГС)**.

**2.1.** Сделайте **простую случайную выборку** из 100 клиентов (`df.sample(n=100, random_state=...)`). Сравните среднее `charges` и долю курильщиков в выборке и в ГС.

**2.2.** Сделайте **стратифицированную выборку** из 100 клиентов, где страты — это `region` (из каждого региона берём пропорционально его доле в ГС; подсказка: `df.groupby("region", group_keys=False).apply(lambda g: g.sample(frac=100/len(df), random_state=...))`). Сравните те же показатели с ГС.

**2.3.** Повторите простую случайную выборку 5 раз с разными `random_state` и посмотрите, как «прыгает» среднее `charges`. Запишите минимальное и максимальное полученное среднее.

**2.4.** Ответьте PM: можно ли принимать решения по одной выборке из 100 клиентов? Какой из двух способов отбора вы бы рекомендовали и почему? Какие ещё страты (кроме региона) имело бы смысл использовать в этой задаче?

In [ ]:
# ваш код: простая случайная выборка и сравнение с ГС


In [ ]:
# ваш код: стратифицированная выборка по региону и сравнение с ГС


In [ ]:
# ваш код: 5 повторов простой случайной выборки, min и max среднего charges


**Вывод:** 

## Задание 3. Портрет клиента: меры центральной тенденции

PM хочет одну цифру на слайд: «средний клиент MedGuard — это …».

**3.1.** Для переменных `age`, `bmi`, `charges` посчитайте **моду, медиану и среднее** (`stats.mode`, `np.median`, `np.mean` или методы pandas). Соберите результат в одну таблицу.

**3.2.** Постройте гистограммы этих трёх переменных (по одной на каждую, `bins=30`). На каждой гистограмме вертикальными линиями отметьте среднее и медиану.

**3.3.** Для каждой переменной ответьте: распределение симметричное или скошенное? Есть ли заметные выбросы / несколько мод? Какую меру центральной тенденции вы **порекомендуете PM для слайда** и почему (опирайтесь на раздел «Примеры использования» из лекции)?

**3.4.** Проверьте свойство среднего: PM хочет показать выплаты **в тысячах долларов**. Создайте колонку `charges_k = charges / 1000` и покажите, что среднее изменилось ровно в 1000 раз, **не пересчитывая среднее заново по данным** — только через свойство $M_{x \cdot c} = c \cdot M_x$.

In [ ]:
# ваш код: мода, медиана, среднее для age, bmi, charges (одна таблица)


In [ ]:
# ваш код: гистограммы с линиями среднего и медианы


In [ ]:
# ваш код: свойство среднего при умножении на константу


**Вывод (какую меру и почему для каждой переменной):** 

## Задание 4. Насколько клиенты разные: меры изменчивости

Финансовый отдел утверждает: «Курильщики — это предсказуемый сегмент, они просто дороже». PM сомневается: дороже — да, но **предсказуемее ли**?

**4.1.** Разбейте клиентов на две группы по `smoker`. Для `charges` в каждой группе посчитайте: размах ($R$), дисперсию ($D$, `ddof=1`), стандартное отклонение ($sd$, `ddof=1`). Не забудьте про поправку $n-1$ — объясните в одном предложении, почему она нужна.

**4.2.** Посчитайте те же показатели **вручную** по формулам из лекции для группы курильщиков (через цикл или векторно: отклонения от среднего → квадраты → сумма → деление на $n-1$ → корень) и убедитесь, что результат совпадает с `np.var` / `np.std`.

**4.3.** Свойство дисперсии. В следующем году все выплаты вырастут на фиксированную величину 500 USD (индексация). Как изменятся $D$ и $sd$ для `charges`? А если выплаты вырастут на 10% (умножение на 1.1)? Ответьте сначала **по свойствам**, затем проверьте кодом.

**4.4.** Ответьте PM: какой сегмент более однородный по выплатам — курильщики или некурящие? Что это значит для планирования бюджета?

In [ ]:
# ваш код: размах, дисперсия, sd по группам smoker


In [ ]:
# ваш код: ручной расчёт дисперсии и sd для курильщиков


In [ ]:
# ваш код: свойства дисперсии (+500 и *1.1)


**Вывод:** 

## Задание 5. Аномально дорогие клиенты: квартили, IQR, box plot

PM: «Есть ощущение, что несколько клиентов «съедают» весь бюджет. Сколько их на самом деле и кто они?»

**5.1.** Для `charges` посчитайте $Q1$, $Q2$ (медиану), $Q3$ и $IQR$ (`np.percentile` или `df.charges.quantile`).

**5.2.** По алгоритму из лекции вычислите границы выбросов: $Q1 - 1.5 \cdot IQR$ и $Q3 + 1.5 \cdot IQR$. Посчитайте, **сколько клиентов** попадают в выбросы и **какую долю** от всех выплат компании они составляют.

**5.3.** Постройте box plot `charges` по группам `smoker` (`sns.boxplot(x="smoker", y="charges", data=df)`), а затем — по `region`. Опишите, что видно.

**5.4.** Посмотрите на «выбросы» из п. 5.2: какая среди них доля курильщиков? Какой у них средний `bmi`? Сравните с базой в целом.

**5.5.** Ответьте PM: это действительно аномалии (ошибки данных, которые надо удалить) или это **закономерный сегмент**? Что вы рекомендуете: удалять их из анализа, анализировать отдельно или оставить как есть? Обоснуйте.

In [ ]:
# ваш код: квартили, IQR, границы выбросов, количество и доля выплат


In [ ]:
# ваш код: box plot по smoker и по region


In [ ]:
# ваш код: портрет клиентов-выбросов (доля курильщиков, средний bmi)


**Вывод:** 

## Задание 6. Нормальное распределение, z-оценки и правило 2–3 сигм

Отдел андеррайтинга хочет ввести **надбавку за высокий BMI**: «клиенты с BMI выше 40 — редкость, их можно оценивать индивидуально». Нужно проверить, насколько это редкость.

**6.1.** Постройте гистограмму `bmi` и наложите на неё кривую плотности нормального распределения с параметрами $\mu = \bar{X}_{bmi}$, $\sigma = sd_{bmi}$ (`stats.norm.pdf`). Похоже ли распределение на нормальное «на глаз»?

**6.2.** Выполните **z-стандартизацию** колонки `bmi` (создайте колонку `bmi_z`). Убедитесь, что новое среднее $\approx 0$, а стандартное отклонение $\approx 1$. Форма гистограммы изменилась?

**6.3.** Клиент с ID = 1317 (последняя строка таблицы, `df.iloc[-1]`) — какова его z-оценка по `bmi`? На сколько стандартных отклонений он отличается от среднего клиента и в какую сторону?

**6.4.** Используя нормальную модель ($\mu$, $\sigma$ из п. 6.1) и `stats.norm.sf` / `.cdf`, посчитайте **теоретическую** долю клиентов с `bmi > 40`. Затем посчитайте **фактическую** долю по данным. Насколько они расходятся?

**6.5.** По правилу 2–3 сигм посчитайте, сколько клиентов должны лежать в интервале $[\mu - 2\sigma; \mu + 2\sigma]$ (ожидается $\approx 95.4\%$), и сравните с фактом.

**6.6.** Ответьте отделу андеррайтинга: справедливо ли утверждение «BMI > 40 — редкость»? Какой порог вы бы предложили, если «редкостью» считать верхние 5% клиентов (подсказка: `stats.norm.isf(0.05, mu, sigma)` и `df.bmi.quantile(0.95)` — сравните оба варианта).

In [ ]:
# ваш код: гистограмма bmi + нормальная кривая


In [ ]:
# ваш код: z-стандартизация bmi


In [ ]:
# ваш код: z-оценка клиента df.iloc[-1]


In [ ]:
# ваш код: теоретическая и фактическая доля bmi > 40; правило 2 сигм; порог 5%


**Вывод:** 

## Задание 7. Центральная предельная теорема на наших данных

Распределение `charges` (задание 3) явно не нормальное. PM спрашивает: «Тогда как вы вообще собираетесь доверять среднему по выборке?» Ответ на это — ЦПТ. Покажите её на практике.

Снова считаем таблицу из 1 338 клиентов **генеральной совокупностью**.

**7.1.** Извлеките из ГС **1000 выборок** по $n = 50$ клиентов каждая (с возвращением, `df.charges.sample(n=50, replace=True)` в цикле) и для каждой посчитайте среднее `charges`. Постройте гистограмму 1000 выборочных средних.

**7.2.** Сравните:
- среднее ГС и среднее выборочных средних;
- стандартное отклонение выборочных средних (это и есть эмпирическая $se$) и теоретическую $se = \dfrac{sd_{ГС}}{\sqrt{n}}$.

**7.3.** Повторите п. 7.1 для $n = 10$ и $n = 200$. Постройте три гистограммы рядом (одинаковая ось X). Как меняется форма и ширина распределения выборочных средних?

**7.4.** Ответьте PM в 2–3 предложениях: почему мы можем использовать среднее выборки и доверительные интервалы, даже если сами выплаты распределены сильно асимметрично? От чего зависит точность?

In [ ]:
# ваш код: 1000 выборок по 50, гистограмма выборочных средних


In [ ]:
# ваш код: сравнение среднего ГС и среднего средних; эмпирическая vs теоретическая se


In [ ]:
# ваш код: n = 10, 50, 200 — три гистограммы


**Вывод:** 

## Задание 8. Доверительные интервалы для бюджета

Финансовый отдел планирует резерв на выплаты. Им нужна не одна цифра, а **диапазон**, в котором с высокой уверенностью лежит средняя выплата на клиента.

**8.1.** Возьмите одну случайную выборку из $n = 100$ клиентов (`random_state=1`). Посчитайте $\bar{X}$, $sd$, $se = sd / \sqrt{n}$ для `charges`.

**8.2.** Постройте **95%** и **99%** доверительные интервалы для среднего `charges`. Коэффициенты 1.96 и 2.58 получите через `stats.norm.isf`, как в лекции, а не «из головы».

**8.3.** Проверьте: попадает ли **истинное** среднее ГС (по всем 1 338 клиентам) в ваши интервалы?

**8.4.** Финансовый отдел готов оплатить сбор данных ещё по 300 клиентам (итого $n = 400$). Не пересчитывая ничего по данным, только через формулу $se$, скажите: во сколько раз сузится 95% ДИ? Затем проверьте на реальной выборке из 400 клиентов.

**8.5.** Постройте 95% ДИ для среднего `charges` **отдельно для курильщиков и некурящих** (по всей базе). Пересекаются ли интервалы? Что это говорит PM ещё до всяких формальных тестов?

**8.6.** Напишите одну фразу для финансового отдела в формате: «С уверенностью 95% средняя выплата на клиента лежит в диапазоне от … до … USD», и объясните в одном предложении, что именно означает «95%» (вспомните вопрос из лекции про правильную интерпретацию ДИ).

In [ ]:
# ваш код: выборка n=100, X_, sd, se


In [ ]:
# ваш код: 95% и 99% ДИ; проверка попадания среднего ГС


In [ ]:
# ваш код: эффект увеличения n до 400


In [ ]:
# ваш код: ДИ для курильщиков и некурящих


**Вывод:** 

## Задание 9. Проверка гипотез: что из «общеизвестного» в компании правда?

Это главная часть работы. В компании ходят четыре утверждения; PM хочет по каждому получить чёткий вердикт: **подтверждаем / отклоняем**, с $p$-значением и с доверительным интервалом.

Для каждой гипотезы оформите ответ по единой схеме (как в лекции, раздел «Идея статистического вывода»):

1. Сформулировать $H_0$ и $H_1$ (двусторонняя или односторонняя альтернатива — обосновать).
2. Посчитать $\bar{X}$, $sd$, $n$, $se = sd/\sqrt{n}$.
3. Посчитать $z = \dfrac{\bar{X} - M_0}{se}$.
4. Посчитать $p$-значение (`stats.norm.cdf` / `.sf`; можно использовать функцию `p_value` из лекции — она ниже).
5. Построить 95% ДИ и проверить, попадает ли в него $M_0$ — результаты двух подходов должны согласоваться.
6. Вердикт при $\alpha = 0.05$ и **вывод для бизнеса** одним-двумя предложениями.

---

### Гипотеза A. «Средний курильщик обходится нам в 30 000 USD в год»
Маркетинг использует эту цифру в презентациях уже два года. Проверьте: равна ли средняя выплата по курильщикам $M_0 = 30\,000$ USD? (Альтернатива — двусторонняя: нас интересует любое отклонение.)

### Гипотеза B. «Наша целевая аудитория — клиенты 39 лет»
Продуктовая стратегия построена вокруг «среднего клиента 39 лет». Проверьте: равен ли средний возраст клиентов $M_0 = 39$ годам?

### Гипотеза C. «Северо-восток укладывается в бюджет 12 500 USD на клиента»
Финансы заложили в бюджет региона `northeast` среднюю выплату $M_0 = 12\,500$ USD. Проверьте, есть ли основания считать, что реальная средняя выплата по этому региону отличается от бюджета.

### Гипотеза D. «Юго-восток — проблемный регион, там выплаты выше 13 000 USD»
Руководитель по региону `southeast` просит увеличить бюджет, утверждая, что средняя выплата там **превышает** $M_0 = 13\,000$ USD. Здесь альтернатива **односторонняя** («больше») — объясните, почему это меняет расчёт $p$-значения.

### Гипотеза E (бонус, ваша собственная).
Сформулируйте **своё** утверждение о клиентской базе, которое было бы полезно PM (например, про `bmi`, `children` или другой регион), и проверьте его по той же схеме.

In [ ]:
# функция из лекции для расчёта p-значения по z-статистике
def p_value(z_stat, alternative="two-sided"):
    if alternative == "two-sided":
        return 2 * (1 - stats.norm.cdf(np.abs(z_stat)))
    if alternative == "less":
        return stats.norm.cdf(z_stat)
    if alternative == "greater":
        return 1 - stats.norm.cdf(z_stat)


def z_test_mean(sample, m0, alternative="two-sided", conf=0.95):
    '''
    Заготовка: верните словарь с n, mean, sd, se, z, p, ci_low, ci_high.
    Заполните тело функции сами — и используйте её для всех гипотез.
    '''
    # ваш код
    pass

### Гипотеза A — «средний курильщик = 30 000 USD»

- $H_0$:
- $H_1$: 

In [ ]:
# ваш код: гипотеза A


**Вердикт и вывод для бизнеса:** 

### Гипотеза B — «средний возраст клиента = 39 лет»

- $H_0$:
- $H_1$: 

In [ ]:
# ваш код: гипотеза B


**Вердикт и вывод для бизнеса:** 

### Гипотеза C — «northeast: средняя выплата = 12 500 USD»

- $H_0$:
- $H_1$: 

In [ ]:
# ваш код: гипотеза C


**Вердикт и вывод для бизнеса:** 

### Гипотеза D — «southeast: средняя выплата > 13 000 USD»

- $H_0$:
- $H_1$:
- Почему односторонняя альтернатива и как это влияет на $p$: 

In [ ]:
# ваш код: гипотеза D


**Вердикт и вывод для бизнеса:**

### Ошибки I и II рода — вопрос без кода

PM спрашивает: «Если мы отклонили гипотезу A и пересчитаем тарифы для курильщиков, чем мы рискуем, если на самом деле маркетинг был прав?»

Опишите, какая это ошибка (I или II рода), и приведите пример **обратной** ошибки для одной из гипотез B–D: что произойдёт с бизнесом, если мы её совершим.

**Ответ:**